In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import vectorbt as vbt
from datetime import datetime, timedelta
import warnings
import contextlib
import tqdm as tqdm
from tabulate import tabulate
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
from datetime import datetime
import hashlib
import itertools
import os
import json 
import mplfinance as mpf
warnings.filterwarnings('ignore') 

In [2]:
data_path = r"C:\Users\OMEN\Desktop\New folder (2)\data\nifty500\nifty500_daily_ohlcv.csv
data = pd.read_csv(data_path, header=[0,1], index_col=0, parse_dates=True)
# 1. Convert index to Datetime
data.index = pd.to_datetime(data.index)

# 2. Drop any rows where the date is missing (prevents KeyError: NaT)
data = data.loc[data.index.notnull()]

# 3. Sort index (required for rolling/shifting logic)
data = data.sort_index()


SyntaxError: unterminated string literal (detected at line 1) (2137109735.py, line 1)

In [ ]:
def calculate_hurst(ts, window=100):
    """Fast Vectorized Hurst Exponent using NumPy sliding windows."""
    vals = ts.values
    n = len(vals)
    hurst_out = np.full(n, 0.5) # Default to 0.5
    
    if n < window:
        return pd.Series(hurst_out, index=ts.index)

    lags = np.arange(2, 20)
    log_lags = np.log(lags)

    # Create sliding window view (memory efficient)
    shape = (n - window + 1, window)
    strides = (vals.strides[0], vals.strides[0])
    windows = np.lib.stride_tricks.as_strided(vals, shape=shape, strides=strides)

    results = []
    # Loop over windows, but perform math in optimized NumPy
    for chunk in windows:
        # Vectorized standard deviation of differences for all lags
        tau = [np.sqrt(np.std(np.subtract(chunk[lag:], chunk[:-lag]))) for lag in lags]
        poly = np.polyfit(log_lags, np.log(tau), 1)
        results.append(poly[0] * 2.0)
    
    hurst_out[window-1:] = results
    return pd.Series(hurst_out, index=ts.index)